# Make PD Phenotypes/GWAS prep

Based on Jeff Kim's pipeline ("Jeff-1_Prep_PD_GWAS_phenotypes")

Jeff suggested me to, before running any new GWASs, run a basic PD GWAS as a "positive control". 
This way, I can be sure I'm running the entire pipeline correctly (for PD, I need to see SNCA as a major signal).

Run the entire pipeline until "PCs" section

1.1 Get demographic information for people with a certain diagnosis/phenotype

1.2 Genetic sex

1.3 Ancestry (genetic)

1.4 PCs

Then, you should be able to run the GWAS pipeline itself (adapt from the following AOU pipeline using Regenie:
https://github.com/all-of-us/ukb-cross-analysis-demo-project/blob/main/aou_workbench_siloed_analyses/06_aou_regenie_gwas.ipynb)

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls gs://vwb-aou-datasets-controlled/v8/
# Note that this data should also be mounted on your workspace

In [ ]:
!ls -l

In [ ]:
# If needed to change directory
import os
os.chdir("../../")

In [ ]:
!wget https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_avx2_20250129.zip
!unzip plink2_linux_avx2_20250129.zip
!chmod 777 plink2

## PD status + Demographic info

IMPORTANT! For the next PD analysis you do with AOU, especially a GWAS, you want to use the "rule of 2":
>You only want to consider as PD the subjects that have been diagnosed in 2 separate consultations

This is because AOU contains data from multiple private sources, so harmonization is trickier.

In [ ]:
import pandas as pd
import os
import numpy as np

In [ ]:
!ls -l

In [ ]:
# Test query
COHORT_QUERY = f"""
-- Python translation of: "Demographics for AoU WGS cohort"
WITH cohort_tbl AS (
    SELECT person_id
    FROM `{os.environ["WORKSPACE_CDR"]}.cb_search_person` cb_search_person
    WHERE cb_search_person.person_id IN (
        SELECT person_id
        FROM `{os.environ["WORKSPACE_CDR"]}.cb_search_person`
        WHERE has_whole_genome_variant = 1
    )
)
SELECT * FROM cohort_tbl
"""

genomic_cohort = pd.read_gbq(
    COHORT_QUERY,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ)
)

genomic_cohort.head()

In [ ]:
## FROM JEFF'S PIPELINE
# This query represents dataset "PD_clinical-sex_age" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_23353493_person_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (381270)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )"""

PD_clinical = pd.read_gbq(
    dataset_23353493_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

PD_clinical.head(5)

In [ ]:
PD_clinical['sex_at_birth'].value_counts()

In [ ]:
## FROM JEFF'S PIPELINE
# This query represents dataset "70andOlder-sex_age" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_92748420_person_sql = """
    SELECT
        person.person_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 70 AND 124 
                AND NOT EXISTS (      SELECT
                    'x'      
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.death` d      
                WHERE
                    d.person_id = p.person_id ) ) )"""

dataset_92748420_person_df = pd.read_gbq(
    dataset_92748420_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_92748420_person_df.head(5)

In [ ]:
## FROM JEFF'S PIPELINE
# This query represents dataset "PD_control_exclusion-idonly" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_60299148_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (43531003, 4101305, 378144, 373747)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (1568287, 1568284, 1568286, 35207355, 1568289, 35207329)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1) 
                        AND is_standard = 0 )) criteria ) )"""

control_exclusion = pd.read_gbq(
    dataset_60299148_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

In [ ]:
## FROM JEFF'S PIPELINE
# This query represents dataset "All_control-sex_age" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_54063587_person_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id"""

controls_notfiltered = pd.read_gbq(
    dataset_54063587_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

controls_notfiltered.head(5)

In [ ]:
controls_filtered = controls_notfiltered[
    ~controls_notfiltered['person_id'].isin(control_exclusion['person_id'])
]
controls_filtered.shape[0]

In [ ]:
controls_filtered['sex_at_birth'].value_counts()

In [ ]:
controls_filtered_sex = controls_filtered[
    controls_filtered['sex_at_birth'].isin(['Female','Male'])
]
controls_filtered_sex.shape[0]

In [ ]:
PD_clinical_sex = PD_clinical[
    PD_clinical['sex_at_birth'].isin(['Female','Male'])
]
PD_clinical_sex.shape[0]

In [ ]:
PD_clinical_sex.dtypes

In [ ]:
PD_clinical_sex['PD'] = '1'
controls_filtered_sex['PD'] = '0'
PHENO_DF = pd.concat([PD_clinical_sex, controls_filtered_sex])

In [ ]:
PHENO_DF['BIRTH_YEAR'] = PHENO_DF['date_of_birth'].dt.year
PHENO_DF['AGE_COV'] = 2024 - PHENO_DF['BIRTH_YEAR']
PHENO_DF['AGE_COV_SQUARED'] = PHENO_DF['AGE_COV']**2
PHENO_DF.head()

In [ ]:
PHENO_DF[
    PHENO_DF['PD'] == '1'
]['AGE_COV'].median(),PHENO_DF[
    PHENO_DF['PD'] == '0'
]['AGE_COV'].median()

In [ ]:
PHENO_DF.head()

## NOTE THAT race and ethnicity here are self-reported. We do not really use those as biological proxies; 
# instead, we use genetic ancestry.

In [ ]:
PHENO_DF.shape[0]

## Genetic sex

In [ ]:
SEX_UPDATE_DF = PHENO_DF[['person_id','sex_at_birth']]
SEX_BIRTH_MALE = SEX_UPDATE_DF['sex_at_birth']=='Male'
SEX_BIRTH_FEMALE = SEX_UPDATE_DF['sex_at_birth']=='Female'
SEX_UPDATE_DF.loc[
    SEX_BIRTH_MALE,
    'SEX_BIRTH_NUM'
] = '1'
SEX_UPDATE_DF.loc[
    SEX_BIRTH_FEMALE,
    'SEX_BIRTH_NUM'
] = '2'
SEX_UPDATE_DF['SEX_BIRTH_NUM'].value_counts()

In [ ]:
SEX_UPDATE_DF = SEX_UPDATE_DF[['person_id','SEX_BIRTH_NUM']]
SEX_UPDATE_DF.columns = ['#IID','SEX']
SEX_UPDATE_DF.head()

In [ ]:
SEX_UPDATE_DF.to_csv('sex_at_birth_for_plink_PD.txt', sep = '\t', index = None)
# this file was moved to one folder above workspace directory, to avoid loss when instance is terminated

In [ ]:
os.getenv("WORKSPACE_BUCKET")

In [ ]:
# Save important files to GCS
#!gsutil -u $GOOGLE_PROJECT cp sex_at_birth_for_plink_PD.txt gs://cloned-mybucket-wb-prompt-kiwi-3077

In [ ]:
!ls -l

In [ ]:
#!gsutil -u $GOOGLE_PROJECT ls gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen/ # imputed genotypes
#/Genomics/controlled/common_variants/EUR/

# Finding array info - bed (can also search directly on workspace)
!gsutil -u $GOOGLE_PROJECT ls -lh gs://vwb-aou-datasets-controlled/v8/microarray/plink/
    
# Finding array info - psam (can also search directly on workspace)
!gsutil -u $GOOGLE_PROJECT ls -lh gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen/

In [ ]:
# For Terra - Copy microarray files to workspace
# NOTE THAT files are too large - so copy them, use them, and consider deleting them afterwards

## IMPORTANT UPDATE! In Verily workbench, you don't need to download those very large files -
# you can simply access them from the original bucket, preventing unecessary payments for data storage
!mkdir array_data

In [ ]:
!ls -l vwb-aou-datasets-controlled/v8/microarray/plink/

In [ ]:
# Remove any potential temporary files
#!rm array_data/exome.chr1.bed_.gstmp

In [ ]:
# Make sure plink is installed and working. If not, go to the beginning of the pipeline and install it.
!./plink2

In [ ]:
# May need to re-pull data from the data bucket
#!gsutil -u $GOOGLE_PROJECT cp gs://pipelines-and-files-wb-prompt-kiwi-3077/*.ipynb ~/workspace/

In [ ]:
# NOTE THAT to run this you need to have AT LEAST 25 gb of RAM and 250gb storage

# Create plink snplist annotation files from bed files
!./plink2 --bfile vwb-aou-datasets-controlled/v8/microarray/plink/arrays \
  --maf 0.01 --geno 0.1 --hwe 1e-15 keep-fewhet \
  --indep-pairwise 1000 100 0.3 \
  --memory 20000 \
  --write-snplist \
  --out array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3

#  --exclude range data/hg38-blacklist.v2.bed \
#  --keep bgen_chr1.psam \

## IMPORTANT! I have moved array_data folder to one above workspace, to prevent data loss

In [ ]:
# Upload important files to bucket, just in case
!gcloud storage cp sex_at_birth_for_plink_PD.txt gs://pipelines-and-files-wb-prompt-kiwi-3077/
# !gcloud storage cp array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3* gs://pipelines-and-files-wb-prompt-kiwi-3077/ # already saved in "Tremor GWAS" pipeline

In [ ]:
!ls -lh array_data
!ls -lh workspace/vwb-aou-datasets-controlled/v8/microarray/plink/

In [ ]:
# NOTE THAT to run this you need to have AT LEAST 25 gb of RAM and 250gb storage
# (highmem setup)

# Remember that, unlike in Terra, you don't need to copy the array files into local storage
# - you can directly access them through the mounted AOU bucket in your workspace.
!./plink2 --bfile workspace/vwb-aou-datasets-controlled/v8/microarray/plink/arrays \
  --extract array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3.prune.in \
  --memory 20000 \
  --make-pgen \
  --out array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars

In [ ]:
# Save important intermediate files
!gcloud storage cp array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars* gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
## For psam files
!./plink2 --pfile array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars \
   --update-sex sex_at_birth_for_plink_PD.txt \
   --make-just-psam \
   --out array_data/array_sex_birth_PD

## For bed files
# !./plink2 \
#   --bfile gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays \ # need to copy files to local storage
#   --update-sex sex_at_birth_for_plink_tremor.txt \
#   --make-just-fam \
#   --out array_sex_birth_tremor

In [ ]:
# Save important intermediate files
!gcloud storage cp array_data/array_sex_birth_PD* gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
import os
os.chdir("../")

!ls -l

In [ ]:
!./plink2 --pfile array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars \
    --split-par hg38 \
    --make-just-pvar \
    --out array_data/array_split_par

In [ ]:
# Save important intermediate files
!gcloud storage cp array_data/array_split_par* gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
!./plink2 --pgen array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.pgen \
    --psam array_data/array_sex_birth_PD.psam \
    --pvar array_data/array_split_par.pvar \
    --maf 0.05 \
    --check-sex max-female-xf=0.25 min-male-xf=0.75 \
    --out array_data/array_sex_check_PD

In [ ]:
# Save important intermediate files
!gcloud storage cp array_data/array_sex_check_PD* gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
!head array_data/array_sex_check_PD.sexcheck

In [ ]:
import pandas as pd

sex_check = pd.read_csv('array_data/array_sex_check_PD.sexcheck', sep = '\t')
sex_check.head()

In [ ]:
sex_check['SNPSEX'].value_counts()

In [ ]:
sex_check[
    sex_check['SNPSEX'].isna()
].shape[0]

In [ ]:
sex_check_ok_only = sex_check[sex_check['STATUS']=='OK']
sex_check_ok_only['GENETIC_SEX'] = 'Female'
sex_check_ok_only.loc[
    sex_check_ok_only['SNPSEX'] == 1,
    'GENETIC_SEX'
] = 'Male'

In [ ]:
PHENO_DF.columns

In [ ]:
PHENO_COV_DF = PHENO_DF[['person_id','AGE_COV','AGE_COV_SQUARED','sex_at_birth','PD']]
PHENO_COV_DF.columns = ['#IID','AGE_COV','AGE_COV_SQUARED','SEX','PD']
PHENO_COV_DF.head()

In [ ]:
PHENO_COV_DF['SEX'].value_counts()

In [ ]:
PHENO_DF['PD'].value_counts()

In [ ]:
PHENO_COV_DF['PD'].value_counts()

## Ancestry
Check Jeff's code - This part is necessary (even though we've subsetted by EUR only) because we don't wan't to completely trust self-reported ancestries

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls -l gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/
#echo_v4_r2.ancestry_preds.tsv

In [ ]:
!gsutil -u $GOOGLE_PROJECT cp \
  gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv \
  .

In [ ]:
ANC = pd.read_csv('echo_v4_r2.ancestry_preds.tsv', sep = '\t')
ANC.head()

In [ ]:
# Save important intermediate files
import pickle
with open("PHENO_COV_DF-PD.pkl", "wb") as f:
    pickle.dump(PHENO_COV_DF, f)
# Originally saved with columns #IID	AGE_COV	AGE_COV_SQUARED	SEX	PD. Still need ANCESTRY and PCs columns.

# To open pickle object
# with open("object.pkl", "rb") as f:
#     obj = pickle.load(f)

!gcloud storage cp PHENO_COV_DF-PD* gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
# OPTIONAL - Reloading PHENO_COV_DF
import pickle

with open("PHENO_COV_DF-PD.pkl", "rb") as f:
    PHENO_COV_DF = pickle.load(f)
PHENO_COV_DF.head()

In [ ]:
ANC = ANC[['research_id', 'ancestry_pred']]
ANC.columns = ['#IID', 'ANCESTRY']

In [ ]:
PHENO_COV_DF_wANC = PHENO_COV_DF.merge(ANC)
PHENO_COV_DF_wANC.shape[0]/PHENO_COV_DF.shape[0]

In [ ]:
PHENO_COV_DF_wANC['ANCESTRY'].value_counts()

In [ ]:
PHENO_COV_DF_wANC[
    PHENO_COV_DF_wANC['PD'] == '1'
]['ANCESTRY'].value_counts()

## PCs

In [ ]:
# Run PCA with PLINK2
#  --bfile workspace/vwb-aou-datasets-controlled/v8/microarray/plink/arrays \  # non-QC data

!./plink2 \
  --pfile array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars \
  --pca approx 20 \
  --out array_data/arrays_PCA

In [ ]:
!gcloud storage cp array_data/arrays_PCA* gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
%pwd
%cd ../../
%pwd

In [ ]:
# First, need to generate "arrays_PCA.eigenvec" and "arrays_PCA.eigenval". Then, read it below

plink_pcs = pd.read_csv('array_data/arrays_PCA.eigenvec', sep = '\s+')
plink_pcs.head()

In [ ]:
PHENO_COV_DF_wANC_wPCs = PHENO_COV_DF_wANC.merge(plink_pcs)
PHENO_COV_DF_wANC_wPCs.shape[0]/PHENO_COV_DF_wANC.shape[0]

In [ ]:
PHENO_COV_DF_wANC_wPCs.head()

In [ ]:
PD_EUR_mask_1 = (PHENO_COV_DF_wANC_wPCs['PD'] == '1') & (PHENO_COV_DF_wANC_wPCs['ANCESTRY'] == 'eur')
PD_EUR_mask_0 = (PHENO_COV_DF_wANC_wPCs['PD'] == '0') & (PHENO_COV_DF_wANC_wPCs['ANCESTRY'] == 'eur')
PHENO_COV_DF_wANC_wPCs.loc[
    PD_EUR_mask_1,
    'PD_EUR'
] = '1'
PHENO_COV_DF_wANC_wPCs.loc[
    PD_EUR_mask_0,
    'PD_EUR'
] = '0'

In [ ]:
PHENO_COV_DF_wANC_wPCs['PD_EUR'].value_counts()

In [ ]:
4/(1/1667+1/215489)  # 1667 - n w/ phenotype; 215489 - n w/o phenotype

In [ ]:
eigenval = pd.read_csv('array_data/arrays_PCA.eigenval', sep = '\s+', header = None)
eigenval.columns = ['eigenval']
eigenval

In [ ]:
all_variance = eigenval['eigenval'].sum()
eigenval['variance_explained'] = eigenval['eigenval']/all_variance
eigenval

In [ ]:
# Save important intermediate files
PHENO_COV_DF_wANC_wPCs.to_csv(
    "pheno_PD_GWAS.tsv",
    sep="\t",
    index=False
)

!gcloud storage cp pheno_PD_GWAS.tsv gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
# After that, you may be able to run the GWAS pipeline itself

In [ ]:
# Extra formatting for regenie
pheno = PHENO_COV_DF_wANC_wPCs.copy()

# Rename #IID to IID
pheno = pheno.rename(columns={"#IID": "IID"})

# Add FID as duplicate of IID
pheno.insert(0, "FID", pheno["IID"])

# Make sure IID is second column
cols = ["FID", "IID"] + [c for c in pheno.columns if c not in ["FID", "IID"]]
pheno = pheno[cols]

pheno.head()

In [ ]:
# Replace empty strings with NA
pheno = pheno.replace(r"^\s*$", np.nan, regex=True)

# Keep only columns used by REGENIE
cols = ["FID", "IID", "PD", "SEX", "AGE_COV"] + [f"PC{i}" for i in range(1, 11)]
pheno_regenie = pheno[cols].copy()

# Make sure phenotype is numeric
pheno_regenie["PD"] = pd.to_numeric(pheno_regenie["PD"], errors="coerce")

# For binary trait, REGENIE expects 0/1 or 1/2 depending settings;
# 0/1 usually works for quantitative unless you specify binary mode.
pheno.head()

In [ ]:
# Save for REGENIE
pheno.to_csv(
    "pheno_PD_GWAS.tsv",
    sep="\t",
    index=False,
    na_rep="NA"
)

!gcloud storage cp pheno_PD_GWAS.tsv gs://pipelines-and-files-wb-prompt-kiwi-3077/

## Extra: Non-pruned variants for Regenie step 2

In [ ]:
!./plink2 \
  --bfile workspace/vwb-aou-datasets-controlled/v8/microarray/plink/arrays \
  --chr 1-22 \
  --maf 0.01 \
  --geno 0.1 \
  --hwe 1e-15 keep-fewhet \
  --make-pgen \
  --memory 20000 \
  --out array_data/arrays_geno_hwe_maf0.01_no_blacklist_fullQC

In [ ]:
!head array_data/arrays_geno_hwe_maf0.01_no_blacklist_fullQC.psam

In [ ]:
# Extra formatting for regenie
!cp array_data/arrays_geno_hwe_maf0.01_no_blacklist_fullQC.psam array_data/arrays_geno_hwe_maf0.01_no_blacklist_fullQC.OLD.psam

!awk 'BEGIN{OFS="\t"} NR==1{print "#FID","IID",$2; next} NR>1{print $1,$1,$2}' \
  array_data/arrays_geno_hwe_maf0.01_no_blacklist_fullQC.psam \
  > array_data/temp.psam

!mv array_data/temp.psam array_data/arrays_geno_hwe_maf0.01_no_blacklist_fullQC.psam

!head array_data/arrays_geno_hwe_maf0.01_no_blacklist_fullQC.psam

In [ ]:
# Extracting variants from WGS files
!ls -lh workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/pgen

In [ ]:
WGS_CHR1_PREFIX = "vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/pgen/clinvar.chr1"

!grep -v -i -E "INFO|FILTER|contig" workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/pgen/clinvar.chr1.pvar | head

In [ ]:
# Check E326K and N370S presence
!grep -i -E "155236376|155235843|E326K|N370S|Asn409Ser" workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/pgen/clinvar.chr1.pvar

In [ ]:
# Create an extract list
#!printf "chr1:155236376\nchr1:155235843\n" > GBA_variants_to_extract.txt

# Extract variants into a new .pgen
!plink2 \
  --pfile workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/pgen/clinvar.chr1 \
  --chr chr1 \
  --from-bp 155235843 \
  --to-bp 155236376 \
  --make-pgen \
  --out array_data/GBA_E326K_N370S_WGS
#--extract GBA_variants_to_extract.txt \

In [ ]:
!grep -v -i -E "INFO|FILTER|contig" array_data/GBA_E326K_N370S_WGS.pvar | head

In [ ]:
!head array_data/GBA_E326K_N370S_WGS.psam

In [ ]:
!awk 'BEGIN{OFS="\t"} NR==1{print "#FID","IID",$2; next} NR>1{print $1,$1,$2}' \
  array_data/GBA_E326K_N370S_WGS.psam \
  > array_data/temp.psam

!mv array_data/temp.psam array_data/GBA_E326K_N370S_WGS.psam

!head array_data/GBA_E326K_N370S_WGS.psam